# aw_07_b4 — Stage B4: P1 champion → PlayWorld SFT (Track B Phase-2, §5.1 / RQ1)

**Parent = P1 champion = B1v2** (`20260807-225109--b1-general-sft-v2--s42--e6e83b`,
sha256:747e5757...; §6 selection recorded in aw_06_b3). B3 (P1 DPO) was a null
transfer result, so the two-stage pipeline proceeds from the SFT champion.

**Design.** B4 is the RQ1 treatment arm: identical PlayWorld SFT budget to A1
(same data seed 1042, same recipe hyperparameters, 2 epochs full budget — NOT
the 200-step probe), differing from A1 in exactly one variable: initialization
(B1v2 P1 adapter vs base). Primary readout: `f_b4_analysis` B4 vs A1 on the
frozen suites (eval-ID + OOD splits), plus B4 vs A2 as the strongest Track-A
comparator.

Cell order: fetch champion → `a_b4_data` → `b_b4_train` → `c_b4_eval` →
`x09f_run_audit` → `f_b4_analysis`.

**Stage record (2026-08-13):** run `20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6`
(parent sha 747e5757..., output sha 378ba470..., train_loss profile healthy, 2ep/250 steps).
Eval run `20260813-000613--eval-playworld--s42--c49f3a`, freeze_fingerprint 3cdcbc30 MATCH,
truncation/runaway 0.0. **RQ1 primary: B4 ≫ A1 on ALL 10 suite×metric cells (p=.0001):**
eval-ID pass +.200 (.3867 vs .1867), template-OOD +.157, comp-OOD +.160, rule-OOD +.127,
adversarial +.250. B4 also ≫ A2 on all cells (p=.0001).

**⚠ OPEN AUDIT (x15): sft_fingerprint drift.** a_b4_data produced
sha256:2764e797... but the aw_05/aw_06 probe builds produced sha256:050f94b2...
(identical CLI/seed; prompt_fingerprint cc2aef0d... matched). The divergence is
confined to the SFT serialization. Run `scripts/x15_sft_data_diff.py` against the
frozen A1-era artifact before treating the RQ1 result as final: multiset-identical
⇒ caveat only; content-diverged ⇒ retrain B4 on the frozen artifact.


**x15 VERDICT (2026-08-13): CONTENT DIVERGED — protocol deviation D1 (v1.3).**
53/2000 records (2.65%) differ between the A1-era frozen artifact and the aw_07
rebuild; prompts identical (cc2aef0d stable across ALL builds), divergence
confined to the oracle demonstration's target choice among equally-optimal BFS
paths → the generator is deterministic only per code commit. All PlayWorld
*evaluations* remain valid (freeze_fingerprint 3cdcbc30 matched on every eval,
incl. A1/A2/probes/B4) and B-track probes all used the same 050f94b2 build, so
B1–B3 conclusions stand. Only the B4-vs-A1 single-variable claim is confounded
(~2.65% oracle tie-break delta). Remediation: x16 resolves the artifact A1
actually trained on (lineage fingerprint match) → retrain **B4v2** on that
frozen artifact → re-run eval/audit/analysis below. v1 B4 run is retained as
the documented deviation baseline.


**x16 RESULT (2026-08-13): both matches NULL.** A1 (f8ea5785) and B4 (2764e797)
each trained on unfrozen in-session builds; none of the 3 persisted HF artifacts
(cc2bd418 / c0cfa17e / 042eb078) matches either. A1's exact bytes are
unrecoverable ⇒ the comparison cannot be restored by retraining B4 alone.
**Remediation (protocol v1.3): pin canonical =
`m97j/aw-playworld:train/v1/playworld_sft.jsonl` (042eb078, current frozen
commit) and retrain BOTH arms — A1v2 and B4v2 — on it (original recipes).**
RQ1 headline rebases to B4v2 vs A1v2; v1 runs stay as deviation baselines.


**v2 STAGE RECORD (CLOSED 2026-08-14).** Canonical artifact pinned
(0f60f9b0 file-sha / 042eb078 lineage-fp). a1v2 run 269d0e (adapter 70c2ecb9),
b4v2 run c56ed2 (adapter d4fcacdd), both on identical bytes; evals a27857/7308ee,
freeze 3cdcbc30 matched. **RQ1 canonical: B4v2 ≫ A1v2 on all 10 cells (p=.0001)** —
eval-ID pass .3933 vs .1667 (+.227), comp-OOD +.167, adversarial +.263.
Sensitivity a1v2-vs-a1v1: all 10 cells n.s. (|Δ|≤.02) → the D1 tie-break drift is
behaviorally benign; the v1 headline replicated under clean provenance.
Secondary observation for the report: A-track arms exhibit the known base-init
termination pathology at eval (a1v2 truncation .994 / runaway .888; scores remain
valid because the verifier parses the leading action block — legal_action_rate
.99 on adversarial), while B4v2 stops cleanly (trunc/runaway 0.0), inheriting the
trained <|im_end|> from B1v2 (§13 v1.2). Two-stage therefore also fixes
termination "for free" — report as a mechanistic advantage, not a confound
(pass verdicts are termination-insensitive).
b4v2-vs-b4v1 sensitivity errored on a missing fetch (see fixed cell) — rerun.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title fetch champion — materialize the B1v2 parent adapter + lineage sha
B1V2_RUN_ID = "20260807-225109--b1-general-sft-v2--s42--e6e83b"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1V2_RUN_ID}
b1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

import json
b1v2_sha = json.load(open(f"runs/{B1V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
print(b1v2_dir, b1v2_sha)


In [ ]:
# @title a_b4_data — deterministic PlayWorld train data + frozen suites (leakage-gated)
# Same generator/seed as A1 (seed 1042, 5 families x 400). Verify the manifest
# sft_fingerprint matches the A1-era value sha256:050f94b2... — any mismatch
# breaks the single-variable design and MUST stop the stage.
!python scripts/build_training_data.py \
  --seed 1042 --scenarios-per-family 400 --output-dir data/train

!python scripts/build_eval_suites.py --episodes-per-suite 300
# freeze_manifest fingerprint MUST equal sha256:3cdcbc30c99e492c... (G3 freeze)


In [ ]:
# @title b_b4_train — PlayWorld SFT from the B1v2 parent (full A1 budget)
!python scripts/run_experiment.py \
  --config configs/experiments/b4_playworld_sft_from_p1.yaml \
  --parent-adapter-dir {b1v2_dir} \
  --override lineage.parent_run_id={B1V2_RUN_ID} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b1 \
  --override lineage.parent_adapter.revision=main \
  --override lineage.parent_adapter.sha256={b1v2_sha} \
  --override data.source.local_path=data/train/playworld_sft.jsonl \
  --hf-sync-repo m97j/aw-runs-b4


In [ ]:
# @title c_b4_eval — B4 adapter on the frozen suites (canonical profile)
B4_RUN_ID = ""  # <- from b_b4_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_RUN_ID}
b4_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b4_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b4


In [ ]:
# @title x09f_run_audit — termination regression check on the B4 eval (CPU)
B4_EVAL = ""  # <- eval run id from c_b4_eval

!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B4_EVAL} --out runs/x09_run_audit_b4.json


In [ ]:
# @title f_b4_analysis — B4 vs A1 (RQ1 primary) and B4 vs A2
A1_EVAL = "20260801-063425--eval-playworld--s42--3bf440"
A2_EVAL = ""  # <- canonical A2 eval run id (from aw_04_a2)

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B4_EVAL} --label-a b4-two-stage \
  --run-b runs/{A1_EVAL} --label-b a1-direct \
  --output runs/{B4_EVAL}/analysis_b4_vs_a1.json --hf-sync-repo m97j/aw-runs-b4

if A2_EVAL:
    !python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2_EVAL} --kind eval
    !python scripts/run_analysis.py \
      --run-a runs/{B4_EVAL} --label-a b4-two-stage \
      --run-b runs/{A2_EVAL} --label-b a2-dpo \
      --output runs/{B4_EVAL}/analysis_b4_vs_a2.json --hf-sync-repo m97j/aw-runs-b4


## Stage checklist — CLOSED 2026-08-14 (one rerun pending)
- [x] Canonical pinned: aw-playworld:train/v1 (file sha 0f60f9b0, lineage-fp 042eb078)
- [x] a1v2 (269d0e, adapter 70c2ecb9) + b4v2 (c56ed2, adapter d4fcacdd) trained on identical bytes
- [x] c_v2_eval freeze gate 3cdcbc30 matched; evals a27857 / 7308ee
- [x] **RQ1 canonical: B4v2 vs A1v2 — 10/10 cells sig p=.0001** (ID pass +.227, adv +.263)
- [x] Sensitivity a1v2-vs-a1v1: all n.s. (D1 drift behaviorally benign; v1 result replicated)
- [x] x09h: b4v2 trunc/runaway 0.0; a1v2 trunc .994/runaway .888 (base-init pathology,
      A-track-wide; verdicts termination-insensitive — report as two-stage side benefit)
- [ ] RERUN: b4v2-vs-b4v1 sensitivity (fetch fix applied) — expect ~null
- [x] RQ1 DONE → proceed: aw_08 B5 (parent b4v2 c56ed2) + A2v2 (parent a1v2 269d0e)


## B4v2 — retrain on the resolved frozen artifact (deviation D1 remediation)


In [ ]:
# @title x16 — resolve which frozen artifact A1 actually trained on (CPU)
A1_RUN_ID = "20260801-030335--a1-playworld-sft--s42--e24d72"
B4_RUN_ID = "20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6"

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_RUN_ID}
!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_RUN_ID}

# List every persisted playworld_sft.jsonl copy here
!python scripts/x16_resolve_sft_provenance.py \
  --candidate m97j/aw-posttrain:train/v1/playworld_sft.jsonl \
  --candidate m97j/aw-playworld:preference_train/v1/playworld_sft.jsonl \
  --candidate m97j/aw-playworld:train/v1/playworld_sft.jsonl \
  --lineage runs/{A1_RUN_ID}/artifacts/lineage.json \
  --lineage runs/{B4_RUN_ID}/artifacts/lineage.json \
  --out runs/x16_sft_provenance.json


## A1v2 + B4v2 — paired retrain on the pinned canonical artifact (D1 remediation, protocol v1.3)

Canonical: `m97j/aw-playworld:train/v1/playworld_sft.jsonl` (lineage-style
fingerprint sha256:042eb078...). Both arms retrain on the SAME bytes under
their original recipes — the RQ1 single variable (initialization) is restored
by construction rather than reconstruction.


In [ ]:
# @title g_v2_data — sha-pinned canonical SFT artifact (both arms)
CANONICAL_SHA = ""  # paste DATASET_SHA256 printed on first run, then keep pinned

!python scripts/fetch_dataset.py \
  --repo m97j/aw-playworld --path train/v1/playworld_sft.jsonl \
  --output data/train/playworld_sft.jsonl --force \
  {"--expected-sha256 " + CANONICAL_SHA if CANONICAL_SHA else ""}


In [ ]:
# @title g_a1v2_retrain — A1 recipe, canonical data (control arm)
!python scripts/run_experiment.py \
  --config configs/experiments/a1_playworld_sft.yaml \
  --override experiment_name=a1v2-playworld-sft \
  --override data.source.local_path=data/train/playworld_sft.jsonl \
  --hf-sync-repo m97j/aw-runs-a1


In [ ]:
# @title g_b4v2_retrain — B4 recipe from B1v2 parent, canonical data (treatment arm)
B1V2_RUN_ID = "20260807-225109--b1-general-sft-v2--s42--e6e83b"
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1V2_RUN_ID}
b1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
import json
b1v2_sha = json.load(open(f"runs/{B1V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]

!python scripts/run_experiment.py \
  --config configs/experiments/b4_playworld_sft_from_p1.yaml \
  --parent-adapter-dir {b1v2_dir} \
  --override lineage.parent_run_id={B1V2_RUN_ID} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b1 \
  --override lineage.parent_adapter.revision=main \
  --override lineage.parent_adapter.sha256={b1v2_sha} \
  --override experiment_name=b4v2-playworld-sft-from-p1 \
  --override data.source.local_path=data/train/playworld_sft.jsonl \
  --hf-sync-repo m97j/aw-runs-b4


In [ ]:
# @title c_v2_eval — both v2 arms on the frozen suites
A1V2_RUN_ID = ""  # <- from g_a1v2_retrain
B4V2_RUN_ID = ""  # <- from g_b4v2_retrain

!python scripts/build_eval_suites.py --episodes-per-suite 300
# freeze_fingerprint MUST equal sha256:3cdcbc30... (G3 gate) — abort otherwise

out = !python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1V2_RUN_ID}
a1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_RUN_ID}
b4v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {a1v2_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-a1
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b4v2_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b4


In [ ]:
# @title x09h + f_v2_analysis — canonical RQ1 table (B4v2 vs A1v2) + sensitivity
A1V2_EVAL = ""  # <- eval run ids printed by c_v2_eval
B4V2_EVAL = ""

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1V2_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{A1V2_EVAL} runs/{B4V2_EVAL} --out runs/x09_run_audit_v2arms.json

# RQ1 canonical
!python scripts/run_analysis.py \
  --run-a runs/{B4V2_EVAL} --label-a b4v2-two-stage \
  --run-b runs/{A1V2_EVAL} --label-b a1v2-direct \
  --output runs/{B4V2_EVAL}/analysis_b4v2_vs_a1v2.json --hf-sync-repo m97j/aw-runs-b4

# Sensitivity: each v2 arm vs its v1 (expected ~null if tie-break drift is benign)
A1_EVAL_V1 = "20260801-063425--eval-playworld--s42--3bf440"
B4_EVAL_V1 = "20260813-000613--eval-playworld--s42--c49f3a"
!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL_V1} --kind eval
# v0.6.10 FIX: this fetch was missing -> FileNotFoundError on the b4v1 suites
!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_EVAL_V1} --kind eval
!python scripts/run_analysis.py \
  --run-a runs/{A1V2_EVAL} --label-a a1v2 --run-b runs/{A1_EVAL_V1} --label-b a1-v1 \
  --output runs/{A1V2_EVAL}/analysis_a1v2_vs_a1v1.json --hf-sync-repo m97j/aw-runs-a1
!python scripts/run_analysis.py \
  --run-a runs/{B4V2_EVAL} --label-a b4v2 --run-b runs/{B4_EVAL_V1} --label-b b4-v1 \
  --output runs/{B4V2_EVAL}/analysis_b4v2_vs_b4v1.json --hf-sync-repo m97j/aw-runs-b4
